<img style="float: center;" src='./assets/jwebbinar.jpg' width="1000px"/> 

## JWebbinar 49
_June 25, 2026_
## Alignment Introduction 

### Author: Armin Rest (STScI), Justin Pierel (STScI)

**Purpose**:<BR>
This notebook contains a basic example of aligning a JWST image to a reference catalog, in this case Gaia. 

**Data**:<BR>
This example is set up to use an example dataset from early JWST NIRCam observations of the LMC, which is illustrative for these purposes. 


<hr style="border:1px solid gray"> </hr>

# Basic Example: The LMC

In [ ]:
# Package imports
from jwst.datamodels import ImageModel
import re,os
import jhat
import matplotlib.pyplot as plt
import os
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.io import fits
from astropy.wcs.utils import skycoord_to_pixel
from astropy import wcs
from astropy.nddata import extract_array
from astropy.visualization import (simple_norm,LinearStretch)


## If you have a CRDS issue, try uncommenting the following three lines
# os.environ["CRDS_PATH"] = os.path.expanduser("~/crds_cache/jwst")
# os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"

# os.makedirs(os.environ["CRDS_PATH"], exist_ok=True)

In [ ]:
input_image='../data/jw01074003001_04101_00002_nrcalong_cal.fits'
outrootdir = '../data/aligned'
outsubdir = 'lmc_example'
telescope = 'jwst'

wcs_align = jhat.st_wcs_align()
verbose=2
wcs_align.verbose=verbose

# first rough cut: best d_rotated+-rough_cut_pix. This is the upper limit for rough_cut
wcs_align.rough_cut_px_min = wcs_align.rough_cut_px_max = 1.5


In [ ]:
wcs_align.set_outbasename(outrootdir=outrootdir,outsubdir=outsubdir,inputname=input_image)
# set the telescope
wcs_align.set_telescope(telescope=telescope,imname=input_image)


### Let's examine the initial alignment relative to Gaia. We select a Gaia star position (you can try other star positions from the saved Gaia catalog after running the below cells). The red crosshair represents the Gaia star position, which we can see is significantly offset in NIRCam

In [ ]:
star_location = SkyCoord(80.53654603023910,-69.51877732495160,unit=u.deg)
ref_fits = fits.open(input_image)
ref_data = ref_fits['SCI',1].data
ref_y,ref_x = skycoord_to_pixel(star_location,wcs.WCS(ref_fits['SCI',1],ref_fits))

ref_cutout = extract_array(ref_data,(51,51),(ref_x,ref_y))

norm1 = simple_norm(ref_cutout,stretch='log',min_cut=-1,max_cut=200)

fig,axes = plt.subplots(1,1)
axes.imshow(ref_cutout, origin='lower',
                      norm=norm1,cmap='gray')
axes.set_title('Pre-Alignment')
axes.tick_params(labelcolor='none',axis='both',color='none')
axes.axvline(25.5,color='r',alpha=.5)
axes.axhline(25.5,color='r',alpha=.5)

plt.show()

### Now we run the photometry routine, which identifies sources in the target image to match to the reference catalog.

In [ ]:
# do the photometry
xshift=0.0 
yshift=0.0 # This can be estimated if necessary by, e.g., comparing images in DS9 or identifying the stellar locus in later figures
wcs_align.phot.verbose = wcs_align.verbose
photfilename = f'{wcs_align.outbasename}.phot.txt'
(photfilename_4check,photcat_loaded) = wcs_align.phot.run_phot(input_image,
                                                          use_dq=True,
                                                          photfilename=photfilename,
                                                          xshift=xshift,
                                                          yshift=yshift,
                                                          overwrite=True)
# get the indices to the detections
ixs = wcs_align.phot.getindices()


### Now comes matching between the target and reference catalogs (here Gaia).

In [ ]:
dmag_max=None
sharpness_lim=(None,None)
roundness1_lim=(-0.75,0.75)
objmag_lim=(17,19)
Nbright=None
refmag_lim = (None,None)

# proper motion
# only for LMC: set pmflag to False
pmflag=False
pm_median=True

# make the initial cut on the image photometry catalog on magnitudes, sharpness, roundness etc
ixs_use = wcs_align.phot.initial_cut_photcat(dmag_max = dmag_max,
                                        sharpness_lim = sharpness_lim, # sharpness limits
                                        roundness1_lim = roundness1_lim, # roundness1 limits 
                                        objmag_lim = objmag_lim, # limits on mag, the magnitude of the objects in the image
                                        Nbright = Nbright,
                                        ixs=ixs)

# Note that refcatname='gaia' will automatically download and match to the Gaia catalog, but you can supply
# a filename here instead to match to your own reference catalog (we will do this in the next section.)
wcs_align.phot.load_and_match_refcat(ixs_obj=ixs_use,
                                refcatname='gaia',
                                refmag_lim=refmag_lim, # limits for initial cut
                                refmagerr_lim=(None,None), # limits for initial cut, needs to be added to options
                                refcolor_lim=(None,None), # limits for initial cut, needs to be added to options
                                pmflag = pmflag, 
                                pm_median=pm_median)

# Save the refcat entries!
refcatfilename = f'{wcs_align.outbasename}.refcat.txt'
print(f'Saving refcat file into {refcatfilename}')
wcs_align.phot.refcat.write(refcatfilename,overwrite=True)



Try iterating with the parameters above to see what happens to the output plots. Increase the magnitude upper limit, the roundness/sharpness parameters, and generally get a feel for what impacts matching.

### Discussion point: Can you visually identify the stellar locus? If you choose a set of parameters that causes a failure in catalog matching, what can you do to improve things?

In [ ]:
# maximum distance between source in image and refcat object, in arcsec 
d2d_max = None
# maximum uncertainty
dmag_max = None
# limits on color of image mag and reference color magnitude
delta_mag_lim =(None,None)
slope_min=-0.008
slope_Nsteps = 20 # slope_max=-slope_min, slope_stepsize=(slope_max-slope_min)/slope_Nsteps
Nfwhm = 2.5
show_initial_plot=1
show_histofit_plots=2
savephottable=True
outbasename=wcs_align.outbasename


ixs_bestmatch= wcs_align.find_good_refcat_matches(ixs=ixs_use,
                                             d2d_max = d2d_max,
                                             delta_mag_lim = delta_mag_lim, # limits on mag-refcat_mainfilter
                                             refmag_lim = refmag_lim, # limits on refcat_mainfilter, the magnitude of the reference catalog
                                             slope_min=slope_min, 
                                             slope_Nsteps = slope_Nsteps, # slope_max=-slope_min, slope_stepsize=(slope_max-slope_min)/slope_Nsteps
                                             show_initial_plot=show_initial_plot,
                                             show_histofit_plots=show_histofit_plots,
                                             savephottable=savephottable,
                                             outbasename=outbasename
                                             )     


### Actually apply the matched catalog to correct the target image.

In [ ]:
showplots = 1
saveplots = 1
savephottable=1
jhatfits = f'{wcs_align.outbasename}_jhat.fits'
(runflag,jhatfits) = wcs_align.run_align2refcat(input_image,
                                           outputfits=jhatfits,
                                           ixs=ixs_bestmatch,
                                           overwrite=True)

#if self.telescope.lower()=='jwst':
wcs_align.update_phottable_final_wcs(jhatfits,
                                    ixs_bestmatch = ixs_bestmatch,
                                    showplots=showplots,
                                    saveplots=saveplots,
                                    savephottable=savephottable,
                                    overwrite=True
                                    )


## You can now rerun the above cells with different parameter choices and see how this impacts the final "after WCS correction" plots. Below is the same star as above, now compared with the aligned image position.

In [ ]:
align_fits = fits.open(jhatfits)
align_data = align_fits['SCI',1].data
align_y,align_x = skycoord_to_pixel(star_location,wcs.WCS(align_fits['SCI',1],align_fits))

align_cutout = extract_array(align_data,(51,51),(align_x,align_y))
norm2 = simple_norm(align_cutout,stretch='log',min_cut=-1,max_cut=200)
fig,axes = plt.subplots(1,2)
axes[0].imshow(ref_cutout, origin='lower',
                      norm=norm1,cmap='gray')
axes[1].imshow(align_cutout, origin='lower',
                      norm=norm2,cmap='gray')
axes[0].set_title('Pre-Alignment')
axes[1].set_title('Aligned')
axes[0].tick_params(labelcolor='none',axis='both',color='none')
axes[1].tick_params(labelcolor='none',axis='both',color='none')
axes[0].axvline(25.5,color='r',alpha=.5)
axes[0].axhline(25.5,color='r',alpha=.5)
axes[1].axvline(25.5,color='r',alpha=.5)
axes[1].axhline(25.5,color='r',alpha=.5)